# Preprocessing functions

In [1]:
import matplotlib as plt
import cv2
import numpy as np

## Denoise

In [2]:
def denoise_image(image, kernel_size=(5, 5), sigma=0, show=False):
    if show:
        denoised = cv2.GaussianBlur(image, kernel_size, sigma)
        plt.imshow(denoised, cmap='gray')
        plt.title('Denoised Image')
        plt.axis('off')
        plt.show()
        return denoised
    else: 
        return cv2.GaussianBlur(image, kernel_size, sigma)

## Binarization

In [3]:
# Função de binarização (thresholding)
def binarize_image(image, threshold=127, max_value=255, method=cv2.THRESH_BINARY, show=False):
    _, binary = cv2.threshold(image, threshold, max_value, method)
    if show:
        plt.imshow(binary, cmap='gray')
        plt.title('Binarized Image')
        plt.axis('off')
        plt.show()
    return binary

## Lowpass filter

In [4]:
def lowpass_filter(image, kernel_size=(5, 5), show=False):
    blurred = cv2.blur(image, kernel_size)
    if show:
        plt.imshow(blurred, cmap='gray')
        plt.title('Low-pass Filtered Image')
        plt.axis('off')
        plt.show()
    return blurred

## Morphological operations

In [5]:
def erode_image(image, kernel_size=(3, 3), iterations=1, show=False):
    kernel = np.ones(kernel_size, np.uint8)
    eroded = cv2.erode(image, kernel, iterations=iterations)
    if show:
        plt.imshow(eroded, cmap='gray')
        plt.title('Eroded Image')
        plt.axis('off')
        plt.show()
    return eroded

def dilate_image(image, kernel_size=(3, 3), iterations=1, show=False):
    kernel = np.ones(kernel_size, np.uint8)
    dilated = cv2.dilate(image, kernel, iterations=iterations)
    if show:
        plt.imshow(dilated, cmap='gray')
        plt.title('Dilated Image')
        plt.axis('off')
        plt.show()
    return dilated

def open_image(image, kernel_size=(3, 3), iterations=1, show=False):
    kernel = np.ones(kernel_size, np.uint8)
    opened = cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel, iterations=iterations)
    if show:
        plt.imshow(opened, cmap='gray')
        plt.title('Opened Image')
        plt.axis('off')
        plt.show()
    return opened

def close_image(image, kernel_size=(3, 3), iterations=1, show=False):
    kernel = np.ones(kernel_size, np.uint8)
    closed = cv2.morphologyEx(image, cv2.MORPH_CLOSE, kernel, iterations=iterations)
    if show:
        plt.imshow(closed, cmap='gray')
        plt.title('Closed Image')
        plt.axis('off')
        plt.show()
    return closed

## Define preprocessing functions into a mapping

In [6]:
# definning wrappers for each preprocessing operation
def identity(img):
    return img

def denoise_wrapper(img):
    return denoise_image(img)

def binarize_wrapper(img):
    return binarize_image(img)

def lowpass_wrapper(img):
    return lowpass_filter(img)

def erode_wrapper(img):
    return erode_image(img)

def dilate_wrapper(img):
    return dilate_image(img)

def open_wrapper(img):
    return open_image(img)

def close_wrapper(img):
    return close_image(img)

preprocessing_methods = {
    'none': identity,
    'denoise': denoise_wrapper,
    'binarize': binarize_wrapper,
    'lowpass': lowpass_wrapper,
    'erode': erode_wrapper,
    'dilate': dilate_wrapper,
    'open': open_wrapper,
    'close': close_wrapper
}

# Preprocessing and Running the Model Workflow

We will now apply each preprocessing operation (denoise, binarization, lowpass filter, morphological operations), as well as the mixing of these operations, to the dataset, then train the Keras model on each preprocessed version, and compare their validation accuracy to determine the best approach for the problem.

In [7]:
def preprocess_images(df, preproc_fn):
    X = []
    y = []
    for idx, row in df.iterrows():
        img_path = row['image_path']
        label = row['label']
        
        if 7 <= label <= 14: # remap labels 7-14 to 0-7
            label = label - 7
        else:
            continue  # skip labels outside 7-14
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"Warning: Could not read {img_path}")
            continue
        processed = preproc_fn(img)
        X.append(processed)
        y.append(label)
    X = np.stack(X)
    y = np.array(y)
    return X, y

## Building the Model and Running the Model

In [8]:
import pandas as pd

train_df = pd.read_csv('train_split.csv')
test_df = pd.read_csv('test_split.csv')

print(f"Train set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")

Train set: 1968 samples
Test set: 492 samples


In [9]:
def build_model(input_shape, num_classes):
    from keras.models import Sequential
    from keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

    model = Sequential([
        Input(shape=input_shape),
        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [10]:
from keras.applications import EfficientNetB0
from keras.models import Model
from keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from keras.optimizers import Adam

def build_state_of_the_art_model(input_shape, num_classes):
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=Input(shape=input_shape))
    base_model.trainable = False  # congela pesos do backbone inicialmente (pode liberar depois para fine-tuning)

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    output = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=output)

    model.compile(optimizer=Adam(learning_rate=1e-4), 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])
    return model

2025-07-06 23:44:05.759849: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-06 23:44:08.044925: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-07-06 23:44:08.045089: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-07-06 23:44:08.470907: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-06 23:44:09.408982: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-06 23:44:09.411424: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [11]:
from itertools import product
import numpy as np
import pandas as pd
import time

start_time = time.time()

results = []
preproc_names = list(preprocessing_methods.keys())

for name in preproc_names:
    if name == 'none':
        continue
    print(f"\n=== Preprocessing: {name} ===")
    X_train, y_train = preprocess_images(train_df, preprocessing_methods[name])
    X_test, y_test = preprocess_images(test_df, preprocessing_methods[name])
    X_train = np.expand_dims(X_train, -1)
    X_test = np.expand_dims(X_test, -1)
    X_train = X_train.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0

    # Run both models
    for model_name, model_fn in [
        ('CustomCNN', build_model),
        ('EfficientNetB0', build_state_of_the_art_model)
    ]:
        if model_name == 'EfficientNetB0':
            X_train_eff = np.repeat(X_train, 3, axis=-1)
            X_test_eff = np.repeat(X_test, 3, axis=-1)
            input_shape = X_train_eff.shape[1:]
            X_tr, X_te = X_train_eff, X_test_eff
        else:
            input_shape = X_train.shape[1:]
            X_tr, X_te = X_train, X_test

        model = model_fn(input_shape=input_shape, num_classes=8)
        history = model.fit(
            X_tr, y_train,
            validation_data=(X_te, y_test),
            epochs=12,
            batch_size=32,
            verbose=2
        )
        val_accuracies = history.history['val_accuracy']
        best_epoch = int(np.argmax(val_accuracies)) + 1
        best_val_acc = float(np.max(val_accuracies))
        results.append({
            'preprocessing': name,
            'model': model_name,
            'best_val_accuracy': best_val_acc,
            'best_epoch': best_epoch
        })
        print(f"Best val accuracy for {name} [{model_name}]: {best_val_acc:.4f} at epoch {best_epoch}")

end_time = time.time()
print(f"Total time for process: {end_time - start_time:.2f} seconds")

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('best_val_accuracy', ascending=False)
results_df


=== Preprocessing: denoise ===
Epoch 1/12
62/62 - 44s - loss: 1.6278 - accuracy: 0.5015 - val_loss: 1.4746 - val_accuracy: 0.5366 - 44s/epoch - 711ms/step
Epoch 2/12
62/62 - 43s - loss: 1.5078 - accuracy: 0.5361 - val_loss: 1.4927 - val_accuracy: 0.5366 - 43s/epoch - 688ms/step
Epoch 3/12
62/62 - 43s - loss: 1.5073 - accuracy: 0.5356 - val_loss: 1.4723 - val_accuracy: 0.5366 - 43s/epoch - 689ms/step
Epoch 4/12
62/62 - 43s - loss: 1.4472 - accuracy: 0.5427 - val_loss: 1.4442 - val_accuracy: 0.5183 - 43s/epoch - 689ms/step
Epoch 5/12
62/62 - 43s - loss: 1.3842 - accuracy: 0.5412 - val_loss: 1.4089 - val_accuracy: 0.5386 - 43s/epoch - 689ms/step
Epoch 6/12
62/62 - 43s - loss: 1.3455 - accuracy: 0.5503 - val_loss: 1.3978 - val_accuracy: 0.5488 - 43s/epoch - 690ms/step
Epoch 7/12
62/62 - 43s - loss: 1.2650 - accuracy: 0.5640 - val_loss: 1.4058 - val_accuracy: 0.5528 - 43s/epoch - 691ms/step
Epoch 8/12
62/62 - 43s - loss: 1.1648 - accuracy: 0.5904 - val_loss: 1.4302 - val_accuracy: 0.5386 -

2025-07-06 23:53:29.797409: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1184956416 exceeds 10% of free system memory.


62/62 - 49s - loss: 1.6524 - accuracy: 0.4964 - val_loss: 1.4758 - val_accuracy: 0.5366 - 49s/epoch - 788ms/step
Epoch 2/12
62/62 - 44s - loss: 1.5449 - accuracy: 0.5198 - val_loss: 1.4741 - val_accuracy: 0.5366 - 44s/epoch - 702ms/step
Epoch 3/12
62/62 - 43s - loss: 1.5335 - accuracy: 0.5285 - val_loss: 1.4715 - val_accuracy: 0.5366 - 43s/epoch - 700ms/step
Epoch 4/12
62/62 - 43s - loss: 1.5307 - accuracy: 0.5300 - val_loss: 1.4719 - val_accuracy: 0.5366 - 43s/epoch - 701ms/step
Epoch 5/12
62/62 - 43s - loss: 1.5547 - accuracy: 0.5325 - val_loss: 1.4817 - val_accuracy: 0.5366 - 43s/epoch - 701ms/step
Epoch 6/12
62/62 - 43s - loss: 1.5244 - accuracy: 0.5340 - val_loss: 1.4761 - val_accuracy: 0.5366 - 43s/epoch - 700ms/step
Epoch 7/12
62/62 - 43s - loss: 1.5286 - accuracy: 0.5361 - val_loss: 1.4741 - val_accuracy: 0.5366 - 43s/epoch - 699ms/step
Epoch 8/12
62/62 - 44s - loss: 1.5282 - accuracy: 0.5340 - val_loss: 1.4781 - val_accuracy: 0.5366 - 44s/epoch - 702ms/step
Epoch 9/12
62/62 - 

2025-07-07 00:10:55.270417: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1184956416 exceeds 10% of free system memory.


62/62 - 49s - loss: 1.6833 - accuracy: 0.4807 - val_loss: 1.4809 - val_accuracy: 0.5366 - 49s/epoch - 786ms/step
Epoch 2/12
62/62 - 43s - loss: 1.5608 - accuracy: 0.5264 - val_loss: 1.4742 - val_accuracy: 0.5366 - 43s/epoch - 698ms/step
Epoch 3/12
62/62 - 43s - loss: 1.5431 - accuracy: 0.5239 - val_loss: 1.4711 - val_accuracy: 0.5366 - 43s/epoch - 698ms/step
Epoch 4/12
62/62 - 43s - loss: 1.5379 - accuracy: 0.5249 - val_loss: 1.4734 - val_accuracy: 0.5366 - 43s/epoch - 698ms/step
Epoch 5/12
62/62 - 43s - loss: 1.5433 - accuracy: 0.5269 - val_loss: 1.4725 - val_accuracy: 0.5366 - 43s/epoch - 700ms/step
Epoch 6/12
62/62 - 43s - loss: 1.5485 - accuracy: 0.5285 - val_loss: 1.4805 - val_accuracy: 0.5366 - 43s/epoch - 698ms/step
Epoch 7/12
62/62 - 43s - loss: 1.5236 - accuracy: 0.5269 - val_loss: 1.4714 - val_accuracy: 0.5366 - 43s/epoch - 700ms/step
Epoch 8/12
62/62 - 43s - loss: 1.5391 - accuracy: 0.5310 - val_loss: 1.4739 - val_accuracy: 0.5366 - 43s/epoch - 698ms/step
Epoch 9/12
62/62 - 

...

2025-07-07 00:45:53.365478: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1184956416 exceeds 10% of free system memory.


62/62 - 50s - loss: 1.6389 - accuracy: 0.4949 - val_loss: 1.4755 - val_accuracy: 0.5366 - 50s/epoch - 805ms/step
Epoch 2/12
62/62 - 44s - loss: 1.5686 - accuracy: 0.5203 - val_loss: 1.4781 - val_accuracy: 0.5366 - 44s/epoch - 711ms/step
Epoch 3/12
62/62 - 44s - loss: 1.5639 - accuracy: 0.5269 - val_loss: 1.4710 - val_accuracy: 0.5366 - 44s/epoch - 705ms/step
Epoch 4/12
62/62 - 44s - loss: 1.5410 - accuracy: 0.5335 - val_loss: 1.4809 - val_accuracy: 0.5366 - 44s/epoch - 704ms/step
Epoch 5/12
62/62 - 44s - loss: 1.5373 - accuracy: 0.5325 - val_loss: 1.4728 - val_accuracy: 0.5366 - 44s/epoch - 704ms/step
Epoch 6/12
62/62 - 44s - loss: 1.5245 - accuracy: 0.5325 - val_loss: 1.4743 - val_accuracy: 0.5366 - 44s/epoch - 704ms/step
Epoch 7/12
62/62 - 44s - loss: 1.5159 - accuracy: 0.5356 - val_loss: 1.4748 - val_accuracy: 0.5366 - 44s/epoch - 705ms/step
Epoch 8/12
62/62 - 44s - loss: 1.5206 - accuracy: 0.5346 - val_loss: 1.4795 - val_accuracy: 0.5366 - 44s/epoch - 704ms/step
Epoch 9/12
62/62 - 

2025-07-07 01:03:48.268945: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1184956416 exceeds 10% of free system memory.


62/62 - 50s - loss: 1.6976 - accuracy: 0.4487 - val_loss: 1.4777 - val_accuracy: 0.5366 - 50s/epoch - 802ms/step
Epoch 2/12
62/62 - 44s - loss: 1.5602 - accuracy: 0.5218 - val_loss: 1.4717 - val_accuracy: 0.5366 - 44s/epoch - 704ms/step
Epoch 3/12
62/62 - 44s - loss: 1.5557 - accuracy: 0.5300 - val_loss: 1.4728 - val_accuracy: 0.5366 - 44s/epoch - 706ms/step
Epoch 4/12
62/62 - 44s - loss: 1.5299 - accuracy: 0.5305 - val_loss: 1.4702 - val_accuracy: 0.5366 - 44s/epoch - 705ms/step
Epoch 5/12
62/62 - 44s - loss: 1.5348 - accuracy: 0.5320 - val_loss: 1.4699 - val_accuracy: 0.5366 - 44s/epoch - 702ms/step
Epoch 6/12
62/62 - 44s - loss: 1.5263 - accuracy: 0.5371 - val_loss: 1.4704 - val_accuracy: 0.5366 - 44s/epoch - 703ms/step
Epoch 7/12
62/62 - 44s - loss: 1.5269 - accuracy: 0.5366 - val_loss: 1.4731 - val_accuracy: 0.5366 - 44s/epoch - 703ms/step
Epoch 8/12
62/62 - 44s - loss: 1.5422 - accuracy: 0.5320 - val_loss: 1.4731 - val_accuracy: 0.5366 - 44s/epoch - 705ms/step
Epoch 9/12
62/62 - 

,preprocessing,model,best_val_accuracy,best_epoch
2,binarize,CustomCNN,0.617886,8
12,close,CustomCNN,0.617886,12
8,dilate,CustomCNN,0.611789,12
10,open,CustomCNN,0.611789,11
6,erode,CustomCNN,0.609756,12
4,lowpass,CustomCNN,0.573171,12
0,denoise,CustomCNN,0.565041,12
1,denoise,EfficientNetB0,0.536585,1
3,binarize,EfficientNetB0,0.536585,1
5,lowpass,EfficientNetB0,0.536585,1


In [12]:
from itertools import product
import numpy as np
import pandas as pd
import time

results = []
preproc_names = list(preprocessing_methods.keys())

start_time = time.time()

# mixed methods (all pairs, both orders, skip identity-identity)
for first, second in product(preproc_names, repeat=2):
    if first == 'none' or second == 'none':
        continue  # skip any one method type combination
    def mixed_fn(img, f=first, s=second):
        return preprocessing_methods[s](preprocessing_methods[f](img))
    print(f"\n=== Preprocessing: {first} -> {second} ===")
    X_train, y_train = preprocess_images(train_df, mixed_fn)
    X_test, y_test = preprocess_images(test_df, mixed_fn)
    X_train = np.expand_dims(X_train, -1)
    X_test = np.expand_dims(X_test, -1)
    X_train = X_train.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0

    for model_name, model_fn in [
        ('CustomCNN', build_model),
        ('EfficientNetB0', build_state_of_the_art_model)
    ]:
        if model_name == 'EfficientNetB0':
            X_train_eff = np.repeat(X_train, 3, axis=-1)
            X_test_eff = np.repeat(X_test, 3, axis=-1)
            input_shape = X_train_eff.shape[1:]
            X_tr, X_te = X_train_eff, X_test_eff
        else:
            input_shape = X_train.shape[1:]
            X_tr, X_te = X_train, X_test

        model = model_fn(input_shape=input_shape, num_classes=8)
        history = model.fit(
            X_tr, y_train,
            validation_data=(X_te, y_test),
            epochs=12,
            batch_size=32,
            verbose=2,
        )
        val_accuracies = history.history['val_accuracy']
        best_epoch = int(np.argmax(val_accuracies)) + 1
        best_val_acc = float(np.max(val_accuracies))
        results.append({
            'preprocessing': f'{first}->{second}',
            'model': model_name,
            'best_val_accuracy': best_val_acc,
            'best_epoch': best_epoch
        })
        print(f"Best val accuracy for {first}->{second} [{model_name}]: {best_val_acc:.4f} at epoch {best_epoch}")

end_time = time.time()
print(f"Total time for process: {end_time - start_time:.2f} seconds")    

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('best_val_accuracy', ascending=False)
results_df


=== Preprocessing: denoise -> denoise ===
Epoch 1/12
62/62 - 44s - loss: 1.5838 - accuracy: 0.5229 - val_loss: 1.4804 - val_accuracy: 0.5366 - 44s/epoch - 713ms/step
Epoch 2/12
62/62 - 43s - loss: 1.5297 - accuracy: 0.5361 - val_loss: 1.4852 - val_accuracy: 0.5366 - 43s/epoch - 696ms/step
Epoch 3/12
62/62 - 43s - loss: 1.5005 - accuracy: 0.5371 - val_loss: 1.4629 - val_accuracy: 0.5366 - 43s/epoch - 699ms/step
Epoch 4/12
62/62 - 43s - loss: 1.4833 - accuracy: 0.5366 - val_loss: 1.4494 - val_accuracy: 0.5366 - 43s/epoch - 698ms/step
Epoch 5/12
62/62 - 43s - loss: 1.4394 - accuracy: 0.5478 - val_loss: 1.4429 - val_accuracy: 0.5224 - 43s/epoch - 695ms/step
Epoch 6/12
62/62 - 43s - loss: 1.4060 - accuracy: 0.5442 - val_loss: 1.4472 - val_accuracy: 0.5203 - 43s/epoch - 695ms/step
Epoch 7/12
62/62 - 43s - loss: 1.3495 - accuracy: 0.5396 - val_loss: 1.4022 - val_accuracy: 0.5427 - 43s/epoch - 697ms/step
Epoch 8/12
62/62 - 43s - loss: 1.3092 - accuracy: 0.5549 - val_loss: 1.4268 - val_accurac

KeyboardInterrupt: 